# Flask CV Extraction Application
## Automatic Resume Parsing with AI-Based Skill Detection and Job Matching

This notebook runs a Flask application that:
1. Extracts text from PDF CVs using `pdfplumber`
2. Detects skills using a Named Entity Recognition (NER) model
3. Matches candidate skills with job requirements
4. Returns job recommendations based on skill matching

**Server Port:** 5002

## 1. Import Required Libraries

In [40]:
import os
import json
import re
import unicodedata
import pandas as pd
import pdfplumber
import torch
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

## 2. Configuration & Setup

Initialize Flask app with CORS support and set up paths to required resources.

In [41]:
# Set up base directory and file paths
BASE_DIR = os.getcwd() 
MODEL_DIR_NAME = "final_model" 
SKILL_CSV = "skills_list.csv"
JOB_JSON = "dataset_job_processed_final.json"

print(f"📂 Working Directory: {BASE_DIR}")

📂 Working Directory: d:\ProjekAkhir\Aplikasi\landing_page\backend


## 3. Load Pre-trained NER Model

Load the fine-tuned BERT model for Named Entity Recognition (NER) to detect skills from text.

In [42]:
# Load NER Model
model_path = os.path.join(BASE_DIR, MODEL_DIR_NAME)
ner_pipeline = None

try:
    if os.path.exists(model_path):
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForTokenClassification.from_pretrained(model_path)
        ner_pipeline = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")
        print("✅ Model AI Loaded.")
except Exception as e:
    print(f"⚠️ Warning: Could not load NER model - {e}")

Device set to use cpu


✅ Model AI Loaded.


## 4. Load Valid Skills Database

Load the skills vocabulary from CSV to validate detected skills.

In [43]:
# Load Valid Skills Set
VALID_SKILLS_SET = set()
skill_file = os.path.join(BASE_DIR, SKILL_CSV)

if os.path.exists(skill_file):
    try:
        try:
            df_skill = pd.read_csv(skill_file, encoding='utf-8', on_bad_lines='skip')
        except:
            df_skill = pd.read_csv(skill_file, encoding='latin1', on_bad_lines='skip')
        
        VALID_SKILLS_SET.update(
            df_skill.iloc[:, 0].dropna().astype(str).str.lower().str.strip().tolist()
        )
        print(f"✅ Skills Loaded: {len(VALID_SKILLS_SET)}")
    except Exception as e:
        print(f"⚠️ Warning: Could not load skills file - {e}")

✅ Skills Loaded: 4479


## 5. Load Job Postings Database

Load job requirements data for matching with candidate skills.

In [44]:
# Load Job Listings
JOB_LIST = []
job_path = os.path.join(BASE_DIR, JOB_JSON)

if os.path.exists(job_path):
    try:
        with open(job_path, "r", encoding="utf-8") as f:
            JOB_LIST = json.load(f)
        print(f"✅ Job Postings Loaded: {len(JOB_LIST)}")
    except Exception as e:
        print(f"⚠️ Warning: Could not load job data - {e}")

✅ Job Postings Loaded: 459


## 6. Utility Functions for Text Processing

Define helper functions for cleaning text, extracting information, and matching skills.

In [45]:
### Function 1: Clean Text
# Remove unicode artifacts, normalize whitespace

def clean_text(text):
    """
    Clean extracted text by removing unicode artifacts and extra whitespace
    
    Args:
        text (str): Raw text to clean
        
    Returns:
        str: Cleaned text
    """
    if not text:
        return ""
    
    # Normalize unicode
    text = unicodedata.normalize("NFKD", text)
    
    # Remove non-ASCII characters
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    
    # Clean whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [46]:
### Function 2: Extract Name from CV
# Heuristic-based name extraction from document text

def extract_name_heuristic(text):
    """
    Extract candidate's name from CV text using heuristics
    
    Args:
        text (str): Full CV text
        
    Returns:
        str: Extracted name or 'Kandidat' if not found
    """
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    skip_words = ["curriculum", "vitae", "resume", "cv", "contact", "email"]
    
    # Check first 6 lines for candidate name
    for line in lines[:6]:
        # Name should be 3-50 chars, not contain skip words, and no email symbols
        if (3 < len(line) < 50 and 
            not any(s in line.lower() for s in skip_words) and 
            "@" not in line):
            return line.title()
    
    return "Kandidat"

In [47]:
### Function 3: Extract GPA and Degree
# Pattern matching for educational credentials

def extract_gpa_degree(text):
    """
    Extract GPA/IPK and degree level from CV text
    
    Args:
        text (str): Full CV text
        
    Returns:
        tuple: (gpa_value, degree_level) - strings or None
    """
    text_lower = text.lower()
    
    # Extract GPA/IPK
    gpa_match = re.search(r"(?:gpa|ipk)\s*[:]?\s*(\d[.,]\d{1,2})", text_lower)
    gpa_val = gpa_match.group(1).replace(",", ".") if gpa_match else None
    
    # Extract Degree using regex patterns with flexibility for spacing and formatting
    degree = None
    
    # Create a version of text without spaces for more flexible matching
    text_no_space = text_lower.replace(" ", "")
    
    # Regex patterns for different degree types (handles spacing and dot variations)
    degree_patterns = [
        (r'\bmaster\b', "Master", text_lower),
        (r'\bbachelor\b', "Bachelor", text_lower),
        (r'\bdiploma\b', "Diploma", text_lower),
        (r'\bsarjana\b', "Sarjana", text_lower),
        (r'\bs\.?\s*kom\b', "S.Kom", text_lower),
        # Flexible pattern for S.Tr.S.D.T - matches with flexible spacing/dots
        (r's[\.\s]*tr[\.\s]*s[\.\s]*d[\.\s]*t', "S.Tr.S.D.T", text_no_space),
        (r'\bs\s*1\b|\bs1\b', "S1", text_lower),
        (r'\bs\s*2\b|\bs2\b', "S2", text_lower),
        (r'\bd\s*3\b|\bd3\b', "D3", text_lower),
    ]
    
    for pattern, label, search_text in degree_patterns:
        if re.search(pattern, search_text):
            degree = label
            break
    
    return gpa_val, degree

In [48]:
### Function 4: Extract Skills Manually
# Regex-based skill matching against known skill database

def extract_skills_manual(text, skill_db):
    """
    Extract skills from text using regex pattern matching
    
    Args:
        text (str): Cleaned CV text
        skill_db (set): Set of known skills (lowercase)
        
    Returns:
        list: List of found skills (title case)
    """
    found = set()
    text_lower = text.lower()
    
    # Match skills with word boundaries
    for skill in skill_db:
        if re.search(r"\b" + re.escape(skill) + r"\b", text_lower):
            found.add(skill.title())
    
    return list(found)

In [49]:
### Function 3.5: Extract Years of Experience
# Extract work experience duration from CV text

def extract_experience_years(text):
    """
    Extract years of experience from CV text
    Looks for patterns like "2+ years", "3 years of experience", etc.
    
    Args:
        text (str): Full CV text
        
    Returns:
        str: Extracted experience (e.g., "2+ years") or None if not found
    """
    text_lower = text.lower()
    
    # Regex patterns to match experience duration
    experience_patterns = [
        r'(\d+)\s*\+?\s*years?\s+(?:of\s+)?experience',  # "2+ years of experience" or "2 years experience"
        r'experience\s*:?\s*(\d+)\s*\+?\s*years?',        # "experience: 2+ years"
        r'(\d+)\s*\+?\s*years?\s+(?:of\s+)?professional', # "2+ years professional"
    ]
    
    for pattern in experience_patterns:
        match = re.search(pattern, text_lower)
        if match:
            years = match.group(1)
            # Check if there's a "+" before the pattern
            full_match = match.group(0)
            if '+' in full_match:
                return f"{years}+ years"
            else:
                return f"{years} years"
    
    return None


In [50]:
### Function 5: Match Jobs Based on Skills
# Score and rank job postings by skill match percentage

def match_jobs(user_skills, jobs):
    """
    Match candidate skills with job requirements and rank jobs
    
    Args:
        user_skills (list): Skills detected from candidate's CV
        jobs (list): List of job postings with requirements
        
    Returns:
        list: Top 50 jobs ranked by match score and matched skill count
    """
    if not user_skills or not jobs:
        return []
    
    user_set = set(s.lower() for s in user_skills)
    ranked = []
    
    for job in jobs:
        # Get job requirements
        reqs = job.get('requirements', "")
        if not reqs:
            continue
        
        # Parse requirements
        req_list = [r.strip() for r in reqs.split(',')]
        job_set = set(r.lower() for r in req_list if r.strip())
        
        # Calculate intersection (matched skills)
        matches = user_set.intersection(job_set)
        match_count = len(matches)
        total_reqs = len(job_set)
        
        # Calculate match score
        if total_reqs > 0:
            score = (match_count / total_reqs) * 100
            
            # Only include if at least 1 skill matches
            if match_count > 0:
                ranked.append({
                    "title": job.get('job_title', 'Unknown'),
                    "company": job.get('company_name', 'Unknown'),
                    "match_score": round(score, 1),
                    "matched_skills": [m.title() for m in matches],
                    "missing_skills": [m.title() for m in (job_set - user_set)]
                })
    
    # Sort by matched skill count and match score (descending)
    ranked.sort(key=lambda x: (len(x['matched_skills']), x['match_score']), reverse=True)
    
    # Return top 50 for pagination
    return ranked[:50]

## 7. Main CV Extraction Function

Extract information from CV PDF files and match with job requirements.

In [51]:
def extract_cv_from_pdf(pdf_file_path):
    """
    Extract information from PDF CV and match with job requirements
    
    Args:
        pdf_file_path (str): Path to the PDF file
        
    Returns:
        dict: Dictionary containing:
        - status: success/error
        - profile: extracted name, degree, gpa, experience
        - skills_detected: list of identified skills
        - job_recommendations: ranked list of matching jobs
        - error: error message if failed
    """
    
    try:
        if not os.path.exists(pdf_file_path):
            return {"status": "error", "error": f"File not found: {pdf_file_path}"}
        
        print(f"\n📄 PROSES FILE: {os.path.basename(pdf_file_path)}")

        # Extract text from PDF
        full_text = ""
        with pdfplumber.open(pdf_file_path) as pdf:
            for page in pdf.pages:
                extracted = page.extract_text()
                if extracted:
                    full_text += extracted + "\n"
        
        # Clean and process text
        cleaned_text = clean_text(full_text)
        name = extract_name_heuristic(full_text)
        gpa, degree = extract_gpa_degree(full_text)
        experience = extract_experience_years(full_text)

        # Extract skills using combined approach (NER + Manual)
        final_skills = set()
        
        # Use NER model if available
        if ner_pipeline:
            try:
                ner_results = ner_pipeline(cleaned_text[:3500])
                for r in ner_results:
                    if r['entity_group'] == 'SKILL':
                        # Clean skill name
                        w = re.sub(r'[^\w\+\#\.]', '', r['word']).strip()
                        if w.lower() in VALID_SKILLS_SET:
                            final_skills.add(w.title())
            except Exception as e:
                print(f"⚠️ NER processing warning: {e}")

        # Manual skill extraction as fallback/supplement
        manual_skills = extract_skills_manual(cleaned_text, VALID_SKILLS_SET)
        final_skills.update(manual_skills)
        final_skills_list = list(final_skills)

        print(f"✓ Skills Found: {len(final_skills_list)}")
        
        # Match with job postings
        recs = match_jobs(final_skills_list, JOB_LIST)
        print(f"✓ Job Recommendations: {len(recs)}")

        return {
            "status": "success",
            "profile": {
                "name": name,
                "degree": degree,
                "gpa": gpa,
                "experience": experience
            },
            "skills_detected": final_skills_list,
            "job_recommendations": recs
        }

    except Exception as e:
        print(f"❌ ERROR: {e}")
        return {"status": "error", "error": str(e)}

## 8. Test & Display Results

Run the CV extraction on a PDF file and display the results directly in the notebook.

In [52]:
# Set path to single PDF file for processing
pdf_path = "../../pdf_cvs/cv_template.pdf"

print("=" * 80)
print("🚀 CV EXTRACTION RESULTS")
print("=" * 80)

# Check if file exists and process it
if os.path.exists(pdf_path):
    # Extract CV information
    result = extract_cv_from_pdf(pdf_path)
    
    if result['status'] == 'success':
        print(f"\n✅ HASIL EKSTRAKSI: {os.path.basename(pdf_path)}")
        print(f"\n📋 PROFIL KANDIDAT:")
        profile = result['profile']
        print(f"   • Nama: {profile['name']}")
        print(f"   • Gelar: {profile['degree']}")
        print(f"   • GPA/IPK: {profile['gpa']}")
        print(f"   • Pengalaman: {profile['experience'] if profile['experience'] else 'Tidak ditemukan'}")
        
        print(f"\n💡 SKILL TERDETEKSI ({len(result['skills_detected'])} items):")
        for skill in result['skills_detected']:
            print(f"   • {skill}")
        
        print(f"\n💼 REKOMENDASI PEKERJAAN ({len(result['job_recommendations'])} cocok):")
        for i, job in enumerate(result['job_recommendations'][:5], 1):
            print(f"\n   {i}. {job['title']} - {job['company']}")
            print(f"      Match Score: {job['match_score']}%")
            print(f"      Matched Skills: {', '.join(job['matched_skills'][:3])}")
            if job['missing_skills']:
                print(f"      Missing Skills: {', '.join(job['missing_skills'][:2])}")
    else:
        print(f"❌ Error processing file: {result.get('error', 'Unknown error')}")
else:
    print(f"⚠️ File not found: {pdf_path}")
    print("\nUSAGE:")
    print("  pdf_path = 'your_cv_file.pdf'")
    print("  result = extract_cv_from_pdf(pdf_path)")
    print("  print(result)")

🚀 CV EXTRACTION RESULTS

📄 PROSES FILE: cv_template.pdf
✓ Skills Found: 14
✓ Job Recommendations: 50

✅ HASIL EKSTRAKSI: cv_template.pdf

📋 PROFIL KANDIDAT:
   • Nama: Bagas Cahya Fajar Bastian
   • Gelar: S.Tr.S.D.T
   • GPA/IPK: 3.86
   • Pengalaman: 2+ years

💡 SKILL TERDETEKSI (14 items):
   • Data Visualization
   • Sql
   • Machine Learning
   • Tableau
   • Raw Data
   • Dashboard
   • Grafana
   • Sql Language
   • R
   • Power Bi
   • Tableau Dashboards
   • Python
   • Data Mining
   • Java

💼 REKOMENDASI PEKERJAAN (50 cocok):

   1. IT AI Roles Job ID 20251219VTC - Supra Primatama Nusantara
      Match Score: 29.6%
      Matched Skills: Sql, Java, Machine Learning
      Missing Skills: Pandas, Tensorflow

   2. DATA ANALYST - DIGITAL TRANSFORMATION & AI - Mega Corindo Asia
      Match Score: 53.8%
      Matched Skills: Sql, Tableau, Python
      Missing Skills: Data Cleaning, Data Analytics

   3. IT Data Scientist - Nabati Group
      Match Score: 46.7%
      Matched Skills

In [1]:
import json
import sqlite3
import pandas as pd
import os

def create_database():
    db_path = 'my_db.db'

    # (Opsional) Hapus database lama jika sudah ada agar data selalu fresh saat script dijalankan
    if os.path.exists(db_path):
        os.remove(db_path)
        print(f"Menghapus database lama: {db_path}")

    # 1. Membuat koneksi ke SQLite
    print("Membuat koneksi ke database...")
    conn = sqlite3.connect(db_path)

    try:
        # 2. Memproses dan memasukkan data skills_list.csv
        print("Memproses tabel 'skills'...")
        skills_df = pd.read_csv('skills_list.csv')
        skills_df.to_sql('skills', conn, if_exists='replace', index=False)

        # 3. Memproses dan memasukkan data list_perguruan_tinggi.csv
        print("Memproses tabel 'perguruan_tinggi'...")
        pt_df = pd.read_csv('list_perguruan_tinggi.csv')
        pt_df.to_sql('perguruan_tinggi', conn, if_exists='replace', index=False)

        # 4. Memproses dan memasukkan data list_program_studi.csv
        print("Memproses tabel 'program_studi'...")
        prodi_df = pd.read_csv('list_program_studi.csv')
        prodi_df.to_sql('program_studi', conn, if_exists='replace', index=False)

        # 5. Memproses dan memasukkan data dataset_job_processed_final.json
        print("Memproses tabel 'jobs'...")
        with open('dataset_job_processed_final.json', 'r', encoding='utf-8') as f:
            jobs_data = json.load(f)

        jobs_df = pd.DataFrame(jobs_data)

        # Penanganan khusus untuk kolom JSON (SQLite tidak mendukung tipe list/dict bawaan)
        # Kita ubah kolom yang berisi list/dict menjadi string JSON
        for col in jobs_df.columns:
            if jobs_df[col].apply(lambda x: isinstance(x, (list, dict))).any():
                jobs_df[col] = jobs_df[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)

        jobs_df.to_sql('jobs', conn, if_exists='replace', index=False)

        print("Semua tabel berhasil dibuat dan data telah dimasukkan!")

    except Exception as e:
        print(f"Terjadi kesalahan: {e}")
        
    finally:
        # 6. Menutup koneksi
        conn.close()
        print("Koneksi database ditutup.")

if __name__ == "__main__":
    create_database()

Membuat koneksi ke database...
Memproses tabel 'skills'...
Memproses tabel 'perguruan_tinggi'...
Memproses tabel 'program_studi'...
Memproses tabel 'jobs'...
Semua tabel berhasil dibuat dan data telah dimasukkan!
Koneksi database ditutup.
